# =============================================================================
# BASELINE DENSENET121 — COMBINED DATASET BINARY CLASSIFICATION
# =============================================================================

Simple binary classification baseline (Genuine=0, Forged=1) trained on
**combined CEDAR + BHSig260 datasets** using a unified 70:15:15 split.

## Model Architecture:

- **Feature Extractor**: DenseNet121 (ImageNet pretrained)
- **Classifier Head**: 2-class binary classification (Genuine vs Forged)
- **Loss**: CrossEntropyLoss

## Training Strategy:

- Single-stage supervised training
- Train on combined data (70%)
- Validate on combined val (15%) for model selection
- Test on combined test (15%) for final evaluation

## Advantages:

- ✅ Simple, interpretable baseline
- ✅ Standard binary classification
- ✅ Direct comparison with complex models
- ✅ Minimal architectural modifications

# =============================================================================
# STEP 1: SETUP, IMPORTS, AND REPRODUCIBILITY
# =============================================================================

In [1]:
import os
import sys
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from PIL import Image

# Set repo root
current_dir = os.path.abspath(os.getcwd())
REPO_ROOT = os.path.abspath(os.path.join(current_dir, '..'))
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

# Import Custom Modules
from models.feature_extractor import DenseNetFeatureExtractor
from utils.model_evaluation import compute_metrics, _plot_det_curve, _plot_far_frr, _plot_confusion_matrix, _plot_score_distribution, _plot_roc_curve
from dataloader.tDCBAM_trainloader import get_baseline_transforms

# Deterministic Seeding
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f" > [System] Seed set to: {seed}")

seed_everything(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" > [System] Device: {DEVICE}")
print(f" > [System] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" > [System] CUDA Device Name: {torch.cuda.get_device_name()}")

/home/lawrence/workspace/thesis/thesis/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


 > [System] Seed set to: 42
 > [System] Device: cuda
 > [System] CUDA Available: True
 > [System] CUDA Device Name: NVIDIA GeForce RTX 5080


# =============================================================================
# STEP 2: CONFIGURATION
# =============================================================================

In [2]:
# Notebook identifier
NOTEBOOK_NAME = 'combined_baseline'
SPLIT_RATIO = '70_15_15'

# Paths
SPLIT_DIR = os.path.join(REPO_ROOT, 'data', 'ratio_splits')
CHECKPOINT_DIR = os.path.join(REPO_ROOT, 'checkpoints', 'combined_baseline')
EVAL_DIR = os.path.join(REPO_ROOT, 'model_evals', 'combined_baseline')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

# Dataset split files
CEDAR_SPLIT_FILE = os.path.join(SPLIT_DIR, f'cedar_split_{SPLIT_RATIO}.json')
BHSIG_BENGALI_SPLIT_FILE = os.path.join(SPLIT_DIR, f'bhsig_bengali_split_{SPLIT_RATIO}.json')
BHSIG_HINDI_SPLIT_FILE = os.path.join(SPLIT_DIR, f'bhsig_hindi_split_{SPLIT_RATIO}.json')

# Training Configuration
EPOCHS = 100
BATCH_SIZE = 30
LR = 1e-3
MOMENTUM = 0.99

# Data Configuration
INPUT_SHAPE = (224, 224)
IMG_SIZE = 224
PREPROCESS = True
AUGMENT = True

print(" > Configuration loaded:")
print(f"   Dataset       : Combined (CEDAR + BHSig260)")
print(f"   Split Ratio   : 70:15:15")
print(f"   Epochs        : {EPOCHS}")
print(f"   Batch Size    : {BATCH_SIZE}")
print(f"   Learning Rate : {LR}")
print(f"   Image Size    : {IMG_SIZE}x{IMG_SIZE}")
print(f"   Checkpoint Dir: {CHECKPOINT_DIR}")
print(f"   Eval Dir      : {EVAL_DIR}")

 > Configuration loaded:
   Dataset       : Combined (CEDAR + BHSig260)
   Split Ratio   : 70:15:15
   Epochs        : 100
   Batch Size    : 30
   Learning Rate : 0.001
   Image Size    : 224x224
   Checkpoint Dir: /home/lawrence/workspace/thesis/thesis/checkpoints/combined_baseline
   Eval Dir      : /home/lawrence/workspace/thesis/thesis/model_evals/combined_baseline


# =============================================================================
# STEP 3: LOAD AND ORGANIZE COMBINED DATASET
# =============================================================================

In [3]:
def merge_splits_from_files(split_files, dataset_names):
    """
    Load pre-generated splits from multiple dataset JSON files and merge them.
    """
    train_dict = {}
    val_dict = {}
    test_dict = {}
    
    for ds_key, split_file in split_files.items():
        if not os.path.exists(split_file):
            print(f"   WARNING: Split file not found: {split_file}")
            continue
        
        with open(split_file, 'r') as f:
            split_data = json.load(f)
        
        ds_label = dataset_names.get(ds_key, ds_key)
        for split_name in ['train', 'val', 'test']:
            if split_name not in split_data:
                continue
            
            users_in_split = split_data[split_name]
            print(f"   {split_name.upper():5s}: {ds_label:20s} -> {len(users_in_split):3d} users")
            
            for uid, user_data in users_in_split.items():
                unique_uid = f"{ds_label}_{uid}"
                if split_name == 'train':
                    train_dict[unique_uid] = user_data
                elif split_name == 'val':
                    val_dict[unique_uid] = user_data
                elif split_name == 'test':
                    test_dict[unique_uid] = user_data
    
    return train_dict, val_dict, test_dict


print(" > Loading Pre-Generated Splits (70:15:15)")
print(f"   Split Directory: {SPLIT_DIR}\n")

split_files = {
    'cedar': CEDAR_SPLIT_FILE,
    'bhsig_bengali': BHSIG_BENGALI_SPLIT_FILE,
    'bhsig_hindi': BHSIG_HINDI_SPLIT_FILE
}

dataset_names = {
    'cedar': 'CEDAR',
    'bhsig_bengali': 'BHSig_Bengali',
    'bhsig_hindi': 'BHSig_Hindi'
}

missing_files = [f for f in split_files.values() if not os.path.exists(f)]
if missing_files:
    print(f"   ERROR: Missing split files:")
    for f in missing_files:
        print(f"      - {f}")
    raise FileNotFoundError(f"Split files not found in {SPLIT_DIR}")

train_dict, val_dict, test_dict = merge_splits_from_files(split_files, dataset_names)

total_users = len(train_dict) + len(val_dict) + len(test_dict)
print(f"\n > COMBINED DATASET: {total_users} users total (70:15:15 split)")
print(f"   Train: {len(train_dict)} users | Val: {len(val_dict)} users | Test: {len(test_dict)} users")

train_gen = sum(len(v.get("genuine", [])) for v in train_dict.values())
train_forg = sum(len(v.get("forged", [])) for v in train_dict.values())
val_gen = sum(len(v.get("genuine", [])) for v in val_dict.values())
val_forg = sum(len(v.get("forged", [])) for v in val_dict.values())
test_gen = sum(len(v.get("genuine", [])) for v in test_dict.values())
test_forg = sum(len(v.get("forged", [])) for v in test_dict.values())

print(f"\n   Train images: {train_gen} genuine + {train_forg} forged = {train_gen + train_forg} total")
print(f"   Val images:   {val_gen} genuine + {val_forg} forged = {val_gen + val_forg} total")
print(f"   Test images:  {test_gen} genuine + {test_forg} forged = {test_gen + test_forg} total")

 > Loading Pre-Generated Splits (70:15:15)
   Split Directory: /home/lawrence/workspace/thesis/thesis/data/ratio_splits

   TRAIN: CEDAR                ->  38 users
   VAL  : CEDAR                ->   8 users
   TEST : CEDAR                ->   9 users
   TRAIN: BHSig_Bengali        ->  70 users
   VAL  : BHSig_Bengali        ->  15 users
   TEST : BHSig_Bengali        ->  15 users
   TRAIN: BHSig_Hindi          -> 112 users
   VAL  : BHSig_Hindi          ->  24 users
   TEST : BHSig_Hindi          ->  24 users

 > COMBINED DATASET: 315 users total (70:15:15 split)
   Train: 220 users | Val: 47 users | Test: 48 users

   Train images: 5280 genuine + 6372 forged = 11652 total
   Val images:   1128 genuine + 1362 forged = 2490 total
   Test images:  1152 genuine + 1386 forged = 2538 total


# =============================================================================
# STEP 4: DATASET CLASS AND TRANSFORMS
# =============================================================================

In [4]:
class BinaryClassificationDataset(Dataset):
    """
    Binary classification dataset (Genuine=0, Forged=1).
    Works with combined CEDAR + BHSig split data.
    """
    def __init__(self, user_dict, transform=None):
        self.samples = []
        self.transform = transform
        
        for uid, data in user_dict.items():
            # Handle both key styles: 'genuine'/'forged' and 'gen'/'forg'
            gen_key = next((k for k in data.keys() if k.lower() in ['genuine', 'gen']), None)
            forg_key = next((k for k in data.keys() if k.lower() in ['forged', 'forgeries', 'forg']), None)
            
            if gen_key:
                for path in data[gen_key]:
                    self.samples.append((path, 0))  # 0 = Genuine
            if forg_key:
                for path in data[forg_key]:
                    self.samples.append((path, 1))  # 1 = Forged
        
        num_gen = sum(1 for _, l in self.samples if l == 0)
        num_forg = sum(1 for _, l in self.samples if l == 1)
        print(f"   Dataset: {len(self.samples)} samples ({num_gen} genuine + {num_forg} forged) from {len(user_dict)} users")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            img = Image.open(img_path).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except Exception as e:
            img = torch.zeros(3, IMG_SIZE, IMG_SIZE)
        return img, label


# Transforms
train_transform, val_transform = get_baseline_transforms(
    input_shape=INPUT_SHAPE, preprocess=PREPROCESS, augment=AUGMENT
)

print(" > Dataset class and transforms defined")

 > Dataset class and transforms defined


# =============================================================================
# STEP 5: TRAINING UTILITIES
# =============================================================================

In [5]:
def initialize_model(device):
    """Initialize DenseNet121 baseline with ImageNet pretrained weights."""
    num_classes = 2
    model = DenseNetFeatureExtractor(
        backbone_name='densenet121', output_dim=num_classes,
        pretrained=True, baseline=True
    ).to(device)
    print(f"   Using ImageNet initialization")
    return model


def train_one_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch, returns avg loss and accuracy."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total


def evaluate_model(model, loader, device, output_dir=None, silent=False):
    """Evaluate model and compute metrics including EER."""
    model.eval()
    all_labels, all_scores = [], []
    
    iter_bar = tqdm(loader, desc="Evaluating", leave=False) if not silent else loader
    
    with torch.no_grad():
        for images, labels in iter_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            all_scores.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    metrics = compute_metrics(all_labels, all_scores)
    
    if output_dir and not silent:
        # Full Reporting
        print(f"\n{'='*10} FINAL TEST RESULTS {'='*10}")
        print(f"EER            : {metrics['eer']:.2%}")
        print(f"AUC            : {metrics['auc']:.4f}")
        print(f"Best Threshold : {metrics.get('threshold', 0):.4f}")
        print(f"Accuracy       : {metrics['accuracy']:.2%}")
        print(f"Precision      : {metrics.get('precision', 0):.2%}")
        print(f"Recall         : {metrics.get('recall', 0):.2%}")
        print(f"F1-Score       : {metrics.get('f1', 0):.2%}")
        print("="*40)
        
        _plot_roc_curve(metrics, output_dir)
        _plot_score_distribution(metrics, output_dir)
        _plot_confusion_matrix(metrics, output_dir)
        _plot_det_curve(metrics, output_dir)
        _plot_far_frr(metrics, output_dir)
    
    return metrics


def train_full_pipeline(model, train_loader, val_loader, device, 
                        epochs, lr, checkpoint_path):
    """
    Train-Val-Test strategy: Use train for optimization, val for model selection.
    Returns best validation metrics dict.
    """
    criterion = nn.CrossEntropyLoss()
    best_eer, best_acc, best_metrics = 1.0, 0.0, {}
    
    # All parameters trainable
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4, betas=(MOMENTUM, MOMENTUM))
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = evaluate_model(model, val_loader, device, silent=True)
        val_eer, val_acc = val_metrics['eer'], val_metrics['accuracy']
        
        print(f"   Epoch {epoch+1:02d}/{epochs} | Loss: {train_loss:.4f} | "
              f"Acc: {train_acc:.2%} | Val EER: {val_eer:.2%} | Val Acc: {val_acc:.2%}")
        scheduler.step(val_eer)
        
        if val_eer < best_eer or (val_eer == best_eer and val_acc > best_acc):
            best_eer, best_acc, best_metrics = val_eer, val_acc, val_metrics
            torch.save({'model_state_dict': model.state_dict(), 'metrics': best_metrics}, checkpoint_path)
            print(f"   >>> Saved! (EER: {val_eer:.2%})")
    
    return best_metrics


print(" > Training utilities defined")

 > Training utilities defined


# =============================================================================
# STEP 6: TRAINING PIPELINE
# =============================================================================

In [6]:
print(f"\n{'='*70}")
print(f"{'BASELINE BINARY CLASSIFICATION — CEDAR + BHSig260 (Combined)':^70}")
print(f"{'='*70}\n")

# Create datasets
print(" > Creating datasets...")
print("   Training set:")
train_dataset = BinaryClassificationDataset(train_dict, transform=train_transform)
print("   Validation set:")
val_dataset = BinaryClassificationDataset(val_dict, transform=val_transform)
print("   Test set:")
test_dataset = BinaryClassificationDataset(test_dict, transform=val_transform)

if len(train_dataset) == 0 or len(val_dataset) == 0 or len(test_dataset) == 0:
    print(f"   ERROR: Empty dataset")
else:
    print(f"\n   Total samples - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=4, pin_memory=True)

# Initialize model
seed_everything(42)
model = initialize_model(DEVICE)

# Train using train set, validate on val set for model selection
print(f"\n{'─'*70}")
print(f"Starting training on combined dataset...")
print(f"{'─'*70}")

checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_combined_baseline.pth")
best_val_metrics = train_full_pipeline(
    model, train_loader, val_loader, DEVICE,
    epochs=EPOCHS, lr=LR,
    checkpoint_path=checkpoint_path
)

# Load best checkpoint and evaluate on test set (NEVER seen during training)
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    state = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state)
    print("\n   Loaded best model from checkpoint")

print(f"\n{'─'*70}")
print(f"Evaluating on test set...")
print(f"{'─'*70}")

final_test_metrics = evaluate_model(model, test_loader, DEVICE, output_dir=EVAL_DIR, silent=False)

print(f"\n{'='*70}")
print("TRAINING COMPLETE")
print(f"{'='*70}\n")

print(f"Model saved to: {checkpoint_path}")
print(f"Evaluation plots saved to: {EVAL_DIR}")


     BASELINE BINARY CLASSIFICATION — CEDAR + BHSig260 (Combined)     

 > Creating datasets...
   Training set:
   Dataset: 11652 samples (5280 genuine + 6372 forged) from 220 users
   Validation set:
   Dataset: 2490 samples (1128 genuine + 1362 forged) from 47 users
   Test set:
   Dataset: 2538 samples (1152 genuine + 1386 forged) from 48 users

   Total samples - Train: 11652, Val: 2490, Test: 2538
 > [System] Seed set to: 42
   Using ImageNet initialization

──────────────────────────────────────────────────────────────────────
Starting training on combined dataset...
──────────────────────────────────────────────────────────────────────


Training:   0%|          | 0/388 [00:00<?, ?it/s]

   Epoch 01/100 | Loss: 0.6206 | Acc: 68.69% | Val EER: 23.49% | Val Acc: 76.51%
   >>> Saved! (EER: 23.49%)


Training:   0%|          | 0/388 [00:00<?, ?it/s]

   Epoch 02/100 | Loss: 0.5715 | Acc: 71.80% | Val EER: 24.29% | Val Acc: 75.70%


...

   Epoch 99/100 | Loss: 0.5208 | Acc: 75.06% | Val EER: 22.16% | Val Acc: 77.83%


Training:   0%|          | 0/388 [00:00<?, ?it/s]

   Epoch 100/100 | Loss: 0.5308 | Acc: 74.22% | Val EER: 22.07% | Val Acc: 77.91%

   Loaded best model from checkpoint

──────────────────────────────────────────────────────────────────────
Evaluating on test set...
──────────────────────────────────────────────────────────────────────


Evaluating:   0%|          | 0/85 [00:00<?, ?it/s]


========== FINAL TEST RESULTS ==========
EER            : 25.26%
AUC            : 0.8227
Best Threshold : 0.5972
Accuracy       : 74.70%
Precision      : 78.05%
Recall         : 74.68%
F1-Score       : 76.33%
 > Saved ROC Plot to: /home/lawrence/workspace/thesis/thesis/model_evals/combined_baseline/roc_curve.png
 > Saved Distribution Plot to: /home/lawrence/workspace/thesis/thesis/model_evals/combined_baseline/score_distribution.png
 > Saved Confusion Matrix to: /home/lawrence/workspace/thesis/thesis/model_evals/combined_baseline/confusion_matrix.png
 > Saved DET Curve to: /home/lawrence/workspace/thesis/thesis/model_evals/combined_baseline/det_curve.png
 > Saved FAR/FRR Plot to: /home/lawrence/workspace/thesis/thesis/model_evals/combined_baseline/far_frr.png

TRAINING COMPLETE

Model saved to: /home/lawrence/workspace/thesis/thesis/checkpoints/combined_baseline/best_combined_baseline.pth
Evaluation plots saved to: /home/lawrence/workspace/thesis/thesis/model_evals/combined_baseline


# =============================================================================
# STEP 7: FINAL RESULTS SUMMARY
# =============================================================================

In [7]:
print(f"\n{'='*70}")
print(f"{'FINAL TEST RESULTS (Baseline — CEDAR + BHSig260 Combined)':^70}")
print(f"{'='*70}")
print(f"EER            : {final_test_metrics['eer']:.4f} ({final_test_metrics['eer']*100:.2f}%)")
print(f"Accuracy       : {final_test_metrics['accuracy']:.4f} ({final_test_metrics['accuracy']*100:.2f}%)")
print(f"AUC            : {final_test_metrics['auc']:.4f}")
print(f"Precision      : {final_test_metrics.get('precision', 0):.4f}")
print(f"Recall         : {final_test_metrics.get('recall', 0):.4f}")
print(f"F1-Score       : {final_test_metrics.get('f1', 0):.4f}")
print(f"Best Threshold : {final_test_metrics.get('threshold', 0):.4f}")
print(f"{'='*70}\n")

# Save results
results = {
    'dataset': 'Combined (CEDAR + BHSig260)',
    'model': 'Baseline DenseNet121',
    'task': 'Binary Classification (Genuine vs Forged)',
    'split': '70:15:15',
    'train_users': len(train_dict),
    'val_users': len(val_dict),
    'test_users': len(test_dict),
    'total_users': total_users,
    'train_samples': len(train_dataset),
    'val_samples': len(val_dataset),
    'test_samples': len(test_dataset),
    'eer': float(final_test_metrics['eer']),
    'accuracy': float(final_test_metrics['accuracy']),
    'auc': float(final_test_metrics['auc']),
    'precision': float(final_test_metrics.get('precision', 0)),
    'recall': float(final_test_metrics.get('recall', 0)),
    'f1': float(final_test_metrics.get('f1', 0)),
    'threshold': float(final_test_metrics.get('threshold', 0))
}

results_path = os.path.join(CHECKPOINT_DIR, 'combined_baseline_results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Results saved to: {results_path}")


      FINAL TEST RESULTS (Baseline — CEDAR + BHSig260 Combined)       
EER            : 0.2526 (25.26%)
Accuracy       : 0.7470 (74.70%)
AUC            : 0.8227
Precision      : 0.7805
Recall         : 0.7468
F1-Score       : 0.7633
Best Threshold : 0.5972

Results saved to: /home/lawrence/workspace/thesis/thesis/checkpoints/combined_baseline/combined_baseline_results.json
